In [0]:
from pyspark.sql import functions as F

staging_order_items = spark.table("brazilian_ecommerce.prod.staging_order_items")
staging_products = spark.table("brazilian_ecommerce.prod.staging_products")
staging_category_translation = spark.table("brazilian_ecommerce.prod.staging_category_translation")
staging_order_reviews = spark.table("brazilian_ecommerce.prod.staging_order_reviews")

# --- attach category (with fallback) to every order item ---
order_items_with_category = (
    staging_order_items
    .join(staging_products.select("product_id", "product_category_name"), on="product_id", how="left")
    .join(staging_category_translation, on="product_category_name", how="left")
    .withColumn(
        "product_category_name_english",
        F.coalesce(
            F.col("product_category_name_english"),
            F.col("product_category_name"),
            F.lit("uncategorized"),
        )
    )
)

# --- identify orders with exactly ONE DISTINCT category ---
single_category_orders = (
    order_items_with_category
    .groupBy("order_id")
    .agg(
        F.countDistinct("product_category_name_english").alias("distinct_category_count"),
        F.first("product_category_name_english").alias("product_category_name_english"),
    )
    .filter(F.col("distinct_category_count") == 1)
    .select("order_id", "product_category_name_english")
)

# --- review scores: single-category orders only ---
gold_category_review_scores = (
    single_category_orders
    .join(staging_order_reviews.select("order_id", "review_score"), on="order_id", how="inner")
    .groupBy("product_category_name_english")
    .agg(
        F.count("review_score").alias("total_reviews"),
        F.avg("review_score").cast("decimal(3,2)").alias("avg_review_score"),
        F.count(F.when(F.col("review_score") <= 2, 1)).alias("low_reviews"),
    )
    .withColumn(
        "pct_low_reviews",
        F.when(
            F.col("total_reviews") > 0,
            (F.col("low_reviews") / F.col("total_reviews")).cast("decimal(5,4)")
        )
    )
    .drop("low_reviews")
    .orderBy(F.desc("total_reviews"))
)

full_table_name = "brazilian_ecommerce.prod.gold_category_review_scores"

(
    gold_category_review_scores.write
    .mode("overwrite")
    .option("comment", "gold table: review score distribution by product category, restricted to single-category orders")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN product_category_name_english SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_gold_category_review_scores PRIMARY KEY (product_category_name_english)
""")

print(f"Loaded {gold_category_review_scores.count()} rows into {full_table_name}")